# 🚀 DCT-GAN Mobile - RunPod Training (RTX 4090)

**Checkpoints separados**: Se guardan en `checkpoints_runpod/` para no mezclarse con Colab

## ⚡ Setup RunPod:
1. **Select GPU**: RTX 4090 ($0.59/hr) o RTX 5090 ($0.89/hr)
2. **Template**: RunPod Pytorch (ya incluye CUDA + PyTorch)
3. **Container Disk**: 20 GB suficiente
4. **Volume**: OPCIONAL (si quieres persistencia, +$0.10/GB/mes)
5. Conectar a Jupyter Lab

**Tiempo estimado**: 2-3 horas (RTX 4090) | **Costo**: ~$1.50-1.80

## 1️⃣ Verificar GPU

In [ ]:
import torch
import sys

print("="*70)
print("VERIFICANDO GPU - RUNPOD")
print("="*70)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    compute_cap = torch.cuda.get_device_capability(0)
    print(f"✅ GPU: {gpu_name}")
    print(f"✅ VRAM: {gpu_memory:.1f} GB")
    print(f"✅ Compute Capability: {compute_cap[0]}.{compute_cap[1]}")
    print(f"✅ CUDA: {torch.version.cuda}")
    print(f"✅ PyTorch: {torch.__version__}")
    
    # Verificar que sea RTX 4090 o mejor
    if 'RTX 4090' in gpu_name or 'RTX 5090' in gpu_name or 'A100' in gpu_name:
        print(f"\n🚀 GPU PERFECTA - Tiempo estimado: 2-3 horas")
    elif 'L4' in gpu_name or 'RTX 4000' in gpu_name:
        print(f"\n⚠️  GPU más lenta - Tiempo estimado: 4-5 horas")
    else:
        print(f"\n⚠️  GPU detectada: {gpu_name} - Verifica rendimiento")
else:
    print("❌ NO HAY GPU - Verifica configuración RunPod")

print(f"\n📍 Python: {sys.version}")
print("="*70)

## 2️⃣ Clonar Repositorio desde GitHub

In [ ]:
import os

# Limpiar si existe
if os.path.exists('DCT-GAN-Mobile'):
    !rm -rf DCT-GAN-Mobile

# Clonar desde GitHub
!git clone https://github.com/jaimelopezm-star/DCT-GAN-Mobile.git

%cd DCT-GAN-Mobile

print("\n✅ Repositorio clonado")
print("📂 Contenido:")
!ls -la

## 3️⃣ Instalar Dependencias

In [ ]:
# RunPod suele tener PyTorch preinstalado, pero verificamos versión
!pip install -q --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q pillow pyyaml tqdm tensorboard

print("✅ Dependencias instaladas/actualizadas")

## 4️⃣ Descargar Tiny ImageNet (200 MB)

In [ ]:
import urllib.request
import zipfile
import shutil
from pathlib import Path
from tqdm import tqdm

print("="*70)
print("DESCARGANDO TINY IMAGENET")
print("="*70)

data_dir = Path("data/imagenet2012")
data_dir.mkdir(parents=True, exist_ok=True)

tiny_url = "http://cs231n.stanford.edu/tiny-imagenet-200.zip"
tiny_zip = data_dir / "tiny-imagenet-200.zip"

print(f"\n📥 Descargando desde Stanford...")

class DownloadProgress(tqdm):
    def update_to(self, b=1, bsize=1, tsize=None):
        if tsize is not None:
            self.total = tsize
        self.update(b * bsize - self.n)

with DownloadProgress(unit='B', unit_scale=True, miniters=1, desc='Descarga') as t:
    urllib.request.urlretrieve(tiny_url, tiny_zip, reporthook=t.update_to)

print("\n📦 Extrayendo...")
with zipfile.ZipFile(tiny_zip, 'r') as zip_ref:
    for member in tqdm(zip_ref.namelist(), desc="Extrayendo"):
        zip_ref.extract(member, data_dir)

tiny_zip.unlink()

print("\n📁 Organizando imágenes...")
tiny_dir = data_dir / "tiny-imagenet-200"
val_src = tiny_dir / "val" / "images"
organized_dir = data_dir / "organized" / "all"
organized_dir.mkdir(parents=True, exist_ok=True)

if val_src.exists():
    for img in tqdm(list(val_src.glob("*.JPEG")), desc="Copiando"):
        shutil.copy2(img, organized_dir / img.name)
    
    count = len(list(organized_dir.glob("*.JPEG")))
    print(f"\n✅ {count} imágenes organizadas")
else:
    print(f"❌ Error: {val_src} no encontrado")

## 5️⃣ Crear Splits (Train/Val/Test)

In [ ]:
import random
from pathlib import Path
import shutil

print("="*70)
print("CREANDO SPLITS (80/10/10)")
print("="*70)

source_dir = Path("data/imagenet2012/organized/all")
base_dir = Path("data/imagenet2012/splits")

images = list(source_dir.glob("*.JPEG"))
print(f"\n📊 Total imágenes: {len(images)}")

random.seed(42)
random.shuffle(images)

# Usar solo 10,000 imágenes (como el paper)
images = images[:10000]

n_train = 8000
n_val = 1000

train_imgs = images[:n_train]
val_imgs = images[n_train:n_train + n_val]
test_imgs = images[n_train + n_val:]

splits = {
    'train': train_imgs,
    'val': val_imgs,
    'test': test_imgs
}

for split_name, split_imgs in splits.items():
    split_dir = base_dir / split_name
    split_dir.mkdir(parents=True, exist_ok=True)
    
    print(f"\nCopiando {len(split_imgs)} imágenes a {split_name}/...")
    for img in tqdm(split_imgs, desc=split_name):
        shutil.copy2(img, split_dir / img.name)

print("\n" + "="*70)
print("SPLITS CREADOS")
print("="*70)
print(f"Train: {len(train_imgs)} imágenes")
print(f"Val:   {len(val_imgs)} imágenes")
print(f"Test:  {len(test_imgs)} imágenes")
print("="*70)

## 6️⃣ OPCIONAL: Montar Google Drive (para backup de checkpoints)

⚠️ **SOLO si estás en Colab o quieres sincronizar**. En RunPod, omitir esta celda.

In [ ]:
# SOLO PARA COLAB - Comentar/omitir en RunPod
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive montado")
except:
    print("ℹ️  No es Colab - Google Drive no disponible (normal en RunPod)")

## 7️⃣ Entrenar - CHECKPOINTS EN checkpoints_runpod/

**IMPORTANTE**: Esta versión guarda checkpoints en carpeta separada `checkpoints_runpod/`

**Configuración**:
- 100 épocas
- Batch size 32
- Checkpoints cada época
- TensorBoard logs en `logs_runpod/`

In [ ]:
import subprocess
import sys
from pathlib import Path

print("="*70)
print("INICIANDO ENTRENAMIENTO - RUNPOD")
print("="*70)

# Crear directorio de checkpoints separado
checkpoint_dir = Path("checkpoints_runpod")
checkpoint_dir.mkdir(exist_ok=True)

log_dir = Path("logs_runpod")
log_dir.mkdir(exist_ok=True)

print(f"\\n📁 Checkpoints se guardarán en: {checkpoint_dir.absolute()}")
print(f"📊 Logs (TensorBoard) en: {log_dir.absolute()}")
print(f"\\n⚠️  SEPARADO de Colab - No habrá conflictos\\n")

# Comando de entrenamiento con rutas personalizadas
cmd = [
    sys.executable,
    "train.py",
    "--config", "configs/base_config.yaml",
    "--checkpoint_dir", str(checkpoint_dir),
    "--log_dir", str(log_dir),
    "--epochs", "100",
    "--batch_size", "32",
    "--device", "cuda"
]

print("Ejecutando comando:")
print(" ".join(cmd))
print("\n" + "="*70)

# Ejecutar
result = subprocess.run(cmd)

if result.returncode == 0:
    print("\n" + "="*70)
    print("✅ ENTRENAMIENTO COMPLETADO")
    print("="*70)
    print(f"\n📁 Checkpoints guardados en: {checkpoint_dir.absolute()}")
    print(f"\nPara descargar checkpoints:")
    print(f"  !zip -r checkpoints_runpod.zip checkpoints_runpod/")
else:
    print("\n❌ ERROR en entrenamiento - Revisa logs arriba")

## 8️⃣ Comprimir y Descargar Checkpoints

In [ ]:
from pathlib import Path
import subprocess

checkpoint_dir = Path("checkpoints_runpod")

if checkpoint_dir.exists():
    checkpoints = list(checkpoint_dir.glob("*.pth"))
    print(f"Encontrados {len(checkpoints)} checkpoints:")
    for ckpt in sorted(checkpoints):
        size_mb = ckpt.stat().st_size / 1024 / 1024
        print(f"  - {ckpt.name} ({size_mb:.1f} MB)")
    
    print("\nComprimiendo...")
    !zip -r checkpoints_runpod.zip checkpoints_runpod/
    
    zip_file = Path("checkpoints_runpod.zip")
    if zip_file.exists():
        size_mb = zip_file.stat().st_size / 1024 / 1024
        print(f"\n✅ Archivo comprimido: checkpoints_runpod.zip ({size_mb:.1f} MB)")
        print(f"\n📥 Descarga desde el explorador de archivos de RunPod")
        print(f"   O usa: files.download('checkpoints_runpod.zip')  # si hay API disponible")
else:
    print("❌ No se encontró carpeta checkpoints_runpod/")

## 9️⃣ Evaluar Modelo Final

In [ ]:
import torch
from pathlib import Path

checkpoint_dir = Path("checkpoints_runpod")
checkpoints = sorted(checkpoint_dir.glob("checkpoint_epoch_*.pth"))

if checkpoints:
    last_ckpt = checkpoints[-1]
    print(f"Cargando último checkpoint: {last_ckpt.name}")
    
    ckpt = torch.load(last_ckpt, map_location='cpu')
    
    print("\n" + "="*70)
    print("MÉTRICAS FINALES")
    print("="*70)
    print(f"Época: {ckpt['epoch']}")
    
    if 'val_metrics' in ckpt:
        metrics = ckpt['val_metrics']
        print(f"\n📊 Validación:")
        print(f"  PSNR: {metrics.get('psnr', 'N/A'):.2f} dB (target: ~58 dB)")
        print(f"  SSIM: {metrics.get('ssim', 'N/A'):.4f} (target: ~0.94)")
        
        if 'psnr' in metrics:
            if metrics['psnr'] >= 55:
                print("\n🎉 ¡EXCELENTE! Métricas cercanas al paper")
            elif metrics['psnr'] >= 45:
                print("\n✅ BUENAS métricas - Ajustar hiperparámetros para mejorar")
            else:
                print("\n⚠️  Métricas por debajo del esperado - Revisar entrenamiento")
    
    print("\n" + "="*70)
else:
    print("❌ No se encontraron checkpoints")

## 🔟 Monitorear con TensorBoard (OPCIONAL)

In [ ]:
# Cargar extensión de TensorBoard
%load_ext tensorboard

# Lanzar TensorBoard
%tensorboard --logdir runs_runpod --port 6007

---

## 📋 Resumen Final

### Archivos generados:
- `checkpoints_runpod/`: Checkpoints de RunPod (SEPARADOS de Colab)
- `runs_runpod/`: TensorBoard logs de RunPod
- `checkpoints_runpod.zip`: Archivo comprimido para descargar

### Para comparar con Colab:
```python
# Checkpoints Colab: checkpoints/
# Checkpoints RunPod: checkpoints_runpod/
# NO SE MEZCLAN - Puedes tener ambos entrenamientos sin conflicto
```

### Siguiente paso:
1. Descargar `checkpoints_runpod.zip`
2. Comparar métricas finales de ambas corridas
3. Usar el mejor modelo

**Tiempo total estimado**: 2-3 horas en RTX 4090  
**Costo estimado**: $1.50-1.80